#Initializations

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

#Read Bronze layer

In [0]:
df = spark.table("workspace.bronze.erp_px_cat_g1v2")

#Silver Transformations

## Trimming leading & trailing spaces

In [0]:
#Instead of reconstructing the DataFrame plan for each column using a for loop, this method updates all string columns in a single plan, making it faster.
#.alias(file.name) - keeps the origianl name of the column instead of "trim(prd_id)" for example

# Old Code:
# for field in df.schema.fields:
#     if isinstance(field.dataType, StringType):
#         df = df.withColumn(field.name, trim(col(field.name)))


df = df.select([
    trim(col(field.name)).alias(field.name) if isinstance(field.dataType, StringType) 
    else col(field.name) 
    for field in df.schema.fields
])

#Normalize Maintenance flag to Boolean

In [0]:
df = df.withColumn(
    "maintenance",
    F.when(F.upper(col("maintenance")) == "YES", F.lit(True))
     .when(F.upper(col("maintenance")) == "NO", F.lit(False))
     .otherwise(None)
)


#Renaming Columns

In [0]:
RENAME_MAP = {
    "id": "category_id",
    "cat": "category",
    "subcat": "subcategory",
    "maintenance": "maintenance_flag"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

#Sanity Checks of dataframe

In [0]:
df.limit(10).display()

#Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_product_category")

#Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.erp_product_category LIMIT 30